## Import required Libraries

In [ ]:
# Imports needed for PySpark processing, Delta operations, Fabric constants, and config setup.
from pyspark.sql.functions import col, current_date, md5, concat_ws, max
from delta.tables import DeltaTable
from com.microsoft.spark.fabric.Constants import Constants
import traceback

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

StatementMeta(, , -1, SessionError, , SessionError)

InvalidHttpRequestToLivy: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.

#### Create Temporary Views of Warehouse Tables

In [ ]:
# Fetches warehouse table info
warehouse_tables_sql = """
    SELECT
        d.source                                        AS source,
        de.storage_name                                 AS storage_name,
        de.schema_name                                  AS schema_name,
        de.table_name                                   AS table_name,
        CONCAT_WS(', ', COLLECT_SET(rb.column_name))    AS column_names
    FROM
        lh_utils.dq_config.data_entity      AS de
    INNER JOIN
        lh_utils.dq_config.rule_binding     AS rb
            ON de.entity_id     = rb.entity_id
            AND rb.is_active    = 1
    INNER JOIN
        lh_utils.dq_config.dataset          AS d
            ON de.dataset_id    = d.id
    WHERE
        LOWER(de.storage_type) = 'warehouse'
    GROUP BY
        d.source,
        de.storage_name,
        de.schema_name,
        de.table_name
"""
warehouse_tables_df = spark.sql(warehouse_tables_sql)

# Iterate through each warehouse table
for row in warehouse_tables_df.collect():
    storage_name = row.storage_name
    schema_name = row.schema_name
    table_name = row.table_name
    column_names = row.column_names if row.column_names else "*"  # Use all columns if none specified
    view_name = f"{storage_name}_{schema_name}_{table_name}_view"
    
    # Read using Synapse SQL
    try:
        spark.read \
            .option(Constants.DatabaseName, storage_name) \
            .synapsesql(f"""
                SELECT
                    *
                FROM
                    {schema_name}.{table_name}
            """).createOrReplaceTempView(view_name)
        print(f"Successfully created view: {view_name}")
    
    except Exception as e:
        print(f"Error processing {schema_name}.{table_name}: {str(e)}")
        traceback.print_exc()

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Load DQ Rules and Delete existing DQ data

In [ ]:
# Loads active DQ rules, joins metadata, and builds dq_config for validation.
"""
    Loads active data quality rules from config tables and prepares for validation:
    1. Fetches rules, conditions, error messages from dq_config tables
    2. Joins with entity metadata and accepted values
    3. Clears previous results from Delta table

    Sources:
    - lh_utils.dq_config.data_entity     (table metadata)
    - lh_utils.dq_config.rule_binding    (active rules)
    - lh_utils.dq_config.dataset         (dataset info)
    - lh_utils.dq_config.accepted_values (validation constraints)

    Output: dq_config DataFrame with all active rules
"""
workspace     = "ws-data-quality"
lh_name       = "lh_utils"
config_schema = "dq_config"
output_table  = "data_quality_errors"
output_schema = "data_governance"

dq_config = spark.sql("""
        SELECT
            d.source                                        AS source,
            de.storage_name                                 AS storage_name,
            LOWER(de.storage_type)                          AS storage_type,
            de.schema_name                                  AS schema_name,
            de.table_name                                   AS table_name,
            de.table_display_name                           AS table_display_name,
            rb.column_name                                  AS column_name,
            rb.column_display_name                          AS column_display_name,
            rb.test_name                                    AS test,
            CASE
                WHEN condition IS NULL OR LOWER(condition)  IN ('', ' ', 'nan', 'none', 'null', 'n/a') THEN '1 = 1'
                ELSE condition
            END                                             AS condition,
            rb.error_message                                AS error_message,
            de.alternate_key_cols                           AS alternate_key_cols,
            de.alternate_key_expr                           AS alternate_key_expr,
            de.primary_key_cols                             AS primary_key_cols,
            de.primary_key_expr                             AS primary_key_expr,
            COALESCE(de.created_date_field, 'CURRENT_DATE') AS created_date_field,
            COALESCE(de.created_by_field  , 'NULL')         AS created_by,
            rb.current_value_expr                           AS current_value_expr,
            av.accepted_values                              AS accepted_values,
            av.unaccepted_values                            AS unaccepted_values,
            av.accepted_min_value                           AS accepted_min_value,
            av.accepted_max_value                           AS accepted_max_value,
            av.unaccepted_min_value                         AS unaccepted_min_value,
            av.unaccepted_max_value                         AS unaccepted_max_value,
            f.regex_exp                                     AS regex_expr,
            rb.rule_type                                    AS rule_type,
            rb.dq_metric                                    AS dq_metric,
            rb.rule_name                                    AS rule_name,
            rb.priority                                     AS priority
        FROM
            lh_utils.dq_config.data_entity      AS de
        LEFT JOIN
            lh_utils.dq_config.rule_binding     AS rb
                ON de.entity_id     = rb.entity_id
        LEFT JOIN
            lh_utils.dq_config.dataset          AS d
                ON de.dataset_id    = d.id
        LEFT JOIN
            lh_utils.dq_config.accepted_values  AS av
                ON av.binding_id    = rb.binding_id
        LEFT JOIN 
             lh_utils.dq_config.formats         AS f
                ON REPLACE(rb.rule_name, ' Check', '') = f.name
        WHERE
            rb.is_active = 1
""")

table_path = f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/{lh_name}.Lakehouse/Tables/{output_schema}/{output_table}"

# Delete table before writing the new data
if DeltaTable.isDeltaTable(spark, table_path):
    DeltaTable.forPath(spark, table_path).delete()
    print("Table Deleted")

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
# Creates fixed-size batches from a list to process items in smaller chunks.
def batch(iterable, n = 10):
    """
    Splits an iterable into fixed-size batches for efficient processing.
    
    Why batch?
    - Avoids single massive SQL queries
    - Processes rules in chunks (default: 10 per query)
    - Prevents query timeouts/performance issues
    """
    l = len(iterable)
    for ndx in range(0, l, n):
        yield iterable[ndx:min(ndx + n, l)]

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Null & Not Null Value Checks

In [ ]:
# Function to validate columns for NULL/NOT NULL rules and write violations to Delta
def null_and_not_checks():
    """
    Validates columns against configured NULL/NOT NULL rules, checking for required presence or absence of null values as specified in data quality configurations.
    - Input: Null & Not Null Checks DQ rules from dq_config table
    - Output: Violation records written to Delta error table
    """

    # Select metadata for Null & Not Null rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols","storage_name","schema_name","table_name","column_name","rule_name","source",
        "alternate_key_expr","primary_key_expr","table_display_name", "column_display_name","test","current_value_expr", "created_by",
        "created_date_field","error_message","rule_type","dq_metric","priority","storage_type","condition"
    ).filter("rule_name IN ('Null Check', 'Not Null Check')").collect()

    # Initialize empty DataFrame for storing results
    results = None
    # Process rules in batches to avoid large queries
    for group in batch(meta, 10):
        # Build SQL query for the batch, checking NULL or NOT NULL as per rule
        null_checks_sql = "\nUNION ALL\n".join([
            f"""
            SELECT
                md5(concat_ws('',
                    CAST({m.alternate_key_cols}     AS STRING),
                    '{m.storage_name}',
                    '{m.schema_name}',
                    '{m.table_name}',
                    '{m.column_name}',
                    '{m.test}',
                    '{m.rule_name}',
                    '{m.source}')
                )                                               AS surrogate_key,
                CAST({m.alternate_key_expr}         AS STRING)  AS alternate_key,
                CAST({m.primary_key_expr}           AS STRING)  AS primary_key,
                '{m.table_display_name}'                        AS table_name,
                '{m.column_display_name}'                       AS column_name,
                '{m.source}'                                    AS source,
                '{m.test}'                                      AS test,
                CAST({m.current_value_expr}         AS STRING)  AS current_value,
                DATE_FORMAT(
                    {m.created_date_field},  'yyyy-MM-dd'
                )                                               AS created_on,
                CAST({m.created_by}                 AS STRING)  AS created_by,
                '{m.error_message}'                             AS error_message,
                '{m.rule_type}'                                 AS rule_type,
                '{m.dq_metric}'                                 AS dq_metric,
                '{m.priority}'                                  AS priority,
                CURRENT_TIMESTAMP()                             AS refreshed_at
            FROM
                {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
            WHERE
                {m.condition}
                AND {m.column_name} {'IS NULL' if m.rule_name == 'Null Check' else 'IS NOT NULL'}
            
        """
            for m in group
        ])

        # Execute SQL and collect results
        df = spark.sql(null_checks_sql)
        # Combine results from this batch with previous batches
        results = df if results is None else results.unionByName(df)

    # Write all violation results to the Delta error table
    write_to_table(results, table_path)
    # Print summary of violations written
    print(f"Wrote", results.count(), "DQ Errors from Null & Not Null Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Accepted & Unaccepted Values Check

In [ ]:
def accepted_and_unaccepted_values_checks():
    """
    Validates columns against configured value rules, checking for allowed/disallowed values as specified in data quality configurations.
    - Input: Accepted/Unaccepted value rules from dq_config table
    - Output: Violation records written to Delta error table
    """
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "source", "created_by",
        "alternate_key_expr", "primary_key_expr", "table_display_name", "column_display_name", "test", "current_value_expr", "created_date_field",
        "error_message", "rule_type", "dq_metric", "priority", "storage_type", "condition", "accepted_values", "unaccepted_values"
    ).filter("rule_name IN ('Accepted Values Check', 'Unaccepted Values Check')").collect()
    
    results = None

    for group in batch(meta, 10):
        accepted_values_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND {m.column_name} {"IN (" + m.unaccepted_values + ")" 
                        if m.rule_name == 'Unaccepted Values Check' and any(x in m.unaccepted_values.upper() for x in ['SELECT', 'FROM']) 
                        else "IN (" + ", ".join("'" + v.strip() + "'" for v in m.unaccepted_values.split(',')) + ")"
                        if m.rule_name == 'Unaccepted Values Check' 
                        else "NOT IN (" + m.accepted_values + ")" 
                        if any(x in m.accepted_values.upper() for x in ['SELECT', 'FROM'])
                        else "NOT IN (" + ", ".join("'" + v.strip() + "'" for v in m.accepted_values.split(',')) + ")"
                    }
            """

            for m in group
        ])

        df = spark.sql(accepted_values_sql)
        results = df if results is None else results.unionByName(df)

    write_to_table(results, table_path)
    print(f"Wrote", results.count(), "DQ Errors from Accepted & Unaccepted Values Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Accepted and Unaccepted String & Sub-string checks

In [ ]:
# Function to validate string columns based on accepted/unaccepted string rules
def accepted_string_check():
    """
    Validates columns against configured string rules, checking for exact/substring matchesas specified in data quality configurations.
    - Input: String validation rules from dq_config table
    - Output: Violation records written to Delta error table
    """

    # Select metadata for string validation rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "source", "created_by",
        "alternate_key_expr", "primary_key_expr", "table_display_name", "column_display_name", "test", "current_value_expr", "created_date_field",
        "error_message", "rule_type", "dq_metric", "priority", "storage_type", "condition","accepted_values", "unaccepted_values"
    ).filter("rule_name IN ('Accepted String Check', 'Unaccepted String Check', 'Accepted Sub-string Check', 'Unaccepted Sub-string Check')").collect()
    
     # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid huge SQL queries
    for group in batch(meta, 10):
        # Build SQL query for the batch based on the specific string rule
        accepted_strings_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND LOWER({m.column_name}) {
                    ' <> ' + "'" + str(m.accepted_values) + "'"
                    if m.rule_name == 'Accepted String Check'
                    else '=' + "'" + str(m.unaccepted_values) + "'"
                    if m.rule_name == 'Unaccepted String Check'
                    else ' NOT LIKE ' + "'%" + str(m.accepted_values) + "%'"
                    if m.rule_name == 'Accepted Sub-string Check'
                    else ' LIKE ' + "'%" + str(m.unaccepted_values) + "%'"
                }
            """

            for m in group
        ])

        # Execute SQL for this batch
        df = spark.sql(accepted_strings_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)
        
    # Write all string validation errors to the Delta error table
    write_to_table(results, table_path)
    # Print summary of violations written
    print(f"Wrote", results.count(), "DQ Errors from Accepted & Unaccepted String Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Accepted & Unaccepted Suffix & Prefix Checks

In [ ]:
# Function to validate string columns for prefix/suffix rules and write violations to Delta
def accepted_prefix_and_suffix_checks():
    """
    Validates columns against configured string rules, checking for prefix/suffix matches as specified in data quality configurations.
    - Input: String validation rules from dq_config table
    - Output: Violation records written to Delta error table
    """

    # Select metadata for prefix/suffix string validation rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "source", "created_by",
        "alternate_key_expr", "primary_key_expr", "table_display_name", "column_display_name", "test", "current_value_expr", "created_date_field",
        "error_message", "rule_type", "dq_metric", "priority", "storage_type", "condition","accepted_values", "unaccepted_values"
    ).filter("rule_name IN ('Accepted Prefix Check', 'Unaccepted Prefix Check', 'Accepted Suffix Check', 'Unaccepted Suffix Check')").collect()
    
    # Process rules in batches to avoid large SQL queries
    results = None

    for group in batch(meta, 10):
        # Build SQL query for each batch applying prefix/suffix checks dynamically
        accepted_strings_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND LOWER({m.column_name}) {
                        ' NOT LIKE ' + "'" + str(m.accepted_values) + "%'"
                        if m.rule_name == 'Accepted Prefix Check'
                        else ' LIKE ' + "'" + str(m.unaccepted_values) + "%'"
                        if m.rule_name == 'Unaccepted Prefix Check'
                        else ' NOT LIKE ' + "'%" + str(m.accepted_values) + "'"
                        if m.rule_name == 'Accepted Suffix Check'
                        else ' LIKE ' + "'%" + str(m.unaccepted_values) + "'"
                        if m.rule_name == 'Unaccepted Suffix Check'
                        else ' IS NULL'
                    }
            """
            for m in group
        ])
        
        # Execute SQL for this batch
        df = spark.sql(accepted_strings_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)
        
    # Write all string validation errors to the Delta error table
    write_to_table(results, table_path)
    # Print summary of violations written
    print(f"Wrote", results.count(), "DQ Errors from Accepted & Unaccepted Suffix & Prefix Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Accepted Range Values Checks

In [ ]:
# Function to validate numeric columns based on accepted/unaccepted range rules
def accepted_range_check():
    # Select metadata for range validation rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "source", "alternate_key_expr", "created_by",
        "primary_key_expr", "table_display_name", "column_display_name", "test","current_value_expr", "created_date_field","error_message", "rule_type", "dq_metric",
        "priority", "storage_type", "condition", "accepted_min_value", "accepted_max_value", "unaccepted_min_value", "unaccepted_max_value"
    ).filter("rule_name IN ('Accepted Min Value Check', 'Accepted Max Value Check', 'Accepted Range Check', 'Unaccepted Range Check')").collect()
    
    # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid large SQL queries
    for group in batch(meta, 10):
        # Build SQL query for the batch applying range/min/max validation dynamically
        accepted_range_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND {m.column_name} {
                        'NOT BETWEEN ' + str(m.accepted_min_value) + ' AND ' + str(m.accepted_max_value) 
                        if m.rule_name == 'Accepted Range Check'
                        else 'BETWEEN ' + str(m.accepted_min_value) + ' AND ' + str(m.accepted_max_value)
                        if m.rule_name == 'Unaccepted Range Check'
                        else '< ' + str(m.accepted_min_value)
                        if m.rule_name == 'Accepted Min Value Check'
                        else '> ' + str(m.accepted_max_value)
                    }
               """

            for m in group
        ])
        # Execute SQL for this batch
        df = spark.sql(accepted_range_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)
    # Write all range validation errors to the Delta error table
    write_to_table(results, table_path)
    # Print summary of violations written
    print(f"Wrote", results.count(), "DQ Errors from Accepted & Unaccepted Range Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Unique and not Unique checks

In [ ]:
# Function to validate uniqueness of columns based on DQ rules
def uniqueness_check():
    """
    Processes 'unique_check' and 'not_unique_check' rules dynamically
    using the same batching and SQL generation approach.
    """

    # Select metadata for uniqueness validation rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "created_by",
        "source", "alternate_key_expr", "primary_key_expr", "table_display_name", "column_display_name", "test", "current_value_expr",
        "created_date_field", "error_message", "rule_type", "dq_metric", "priority", "storage_type", "condition"
    ).filter("rule_name IN ('Uniqueness Check', 'Not Uniqueness Check')").collect()

    # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid large SQL queries
    for group in batch(meta, 10):
        # Build SQL query for each batch to check uniqueness
        uniqueness_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(CONCAT(
                        CONCAT_WS(CHAR(44),CAST(ANY_VALUE({m.alternate_key_cols}) AS STRING)),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}')
                    )                                           AS surrogate_key,
                    CONCAT_WS(CHAR(44),
                        CONCAT_WS(',', COLLECT_SET(CAST({m.alternate_key_expr}         AS STRING)))
                    )                                           AS alternate_key,

                    CONCAT_WS(CHAR(44),
                        CONCAT_WS(',', COLLECT_SET(CAST({m.primary_key_expr} AS STRING)))
                    )                                           AS primary_key,
                    '{m.table_display_name}'                    AS table_name,
                    '{m.column_display_name}'                   AS column_name,
                    '{m.source}'                                AS source,
                    '{m.test}'                                  AS test,
                    CAST(
                        ANY_VALUE({m.current_value_expr}
                    )                               AS STRING)  AS current_value,
                    DATE_FORMAT(
                        MAX({m.created_date_field}), 'yyyy-MM-dd'
                    )                                           AS created_on,
                    CAST(MAX({m.created_by})         AS STRING) AS created_by,
                    '{m.error_message}'                         AS error_message,
                    '{m.rule_type}'                             AS rule_type,
                    '{m.dq_metric}'                             AS dq_metric,
                    '{m.priority}'                              AS priority,
                    CURRENT_TIMESTAMP()                         AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND {m.column_name} IS NOT NULL
                GROUP BY
                    {m.column_name}
                HAVING
                    COUNT(*) {' > 1' if m.rule_name == 'Uniqueness Check' else ' = 1'}
            """
            for m in group
        ])

        # Execute SQL for this batch
        df = spark.sql(uniqueness_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)

    # Write all uniqueness validation errors to the Delta error table if any
    if results:
        write_to_table(results, table_path)
        print(f"Wrote", results.count(), "DQ Errors from Uniquness Checks")
    else:
        print("No uniqueness issues found.")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Format checks

In [ ]:
# Function to validate columns against regex-based format rules
def format_checks():
    """
    Handles all format_* rules using regex patterns from metadata.
    Validates data that must match a given regex.
    """

    # Select metadata for format validation rules from dq_config
    meta = dq_config.select(
        "alternate_key_cols","storage_name","schema_name","table_name","column_name","rule_name","source",
        "alternate_key_expr","primary_key_expr","table_display_name", "column_display_name","test","current_value_expr", "created_by",
        "created_date_field","error_message","rule_type","dq_metric","priority","storage_type","condition", "regex_expr"
    ).filter("rule_name LIKE '% Format Check'").collect()

    # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid large SQL queries
    for group in batch(meta, 10):
        format_checks_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND {m.column_name} IS NOT NULL
                    AND NOT REGEXP_LIKE({m.column_name}, '{m.regex_expr}')
            """

            for m in group
        ])

        # print(format_checks_sql)
        df = spark.sql(format_checks_sql)
        results = df if results is None else results.unionByName(df)

    # Write all uniqueness validation errors to the Delta error table if any
    if results:
        write_to_table(results, table_path)
        print(f"Wrote", results.count(), "DQ Errors from Format Checks")
    else:
        print("No format issues found.")

StatementMeta(, , -1, Cancelled, , Cancelled)

### Non-Negative & Non-Positive value checks

In [ ]:
# Function to validate non-negative, non-positive, zero, and non-zero values
def non_negative_and_positive_values_checks():
    """
    Handles rules:
    - non_negative_value: value must be >= accepted_min_value (usually 0)
    - non_positive_value: value must be <= accepted_max_value (usually 0)
    """

    # Select metadata for numeric value checks from dq_config
    meta = dq_config.select(
        "alternate_key_cols","storage_name","schema_name","table_name","column_name","rule_name","source",
        "alternate_key_expr","primary_key_expr","table_display_name", "column_display_name","test","current_value_expr", "created_by",
        "created_date_field","error_message","rule_type","dq_metric","priority","storage_type","condition"
    ).filter("rule_name IN ('Non Negative Value Check', 'Non Positive Value Check', 'Zero Value Check', 'Non-zero Value Check')").collect()

    # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid large SQL queries
    for group in batch(meta, 10):
        # Build SQL query for the batch applying numeric validation rules
        non_negative_and_positive_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND {m.column_name} {
                             '< 0'  if m.rule_name == 'Non Negative Value Check'
                        else '> 0'  if m.rule_name == 'Non Positive Value Check'
                        else '= 0'  if m.rule_name == 'Zero Value Check'
                        else '!= 0' if m.rule_name == 'Non-zero Value Check'
                        else 'IS NULL'
                    }
            """
            for m in group
        ])

        # Execute SQL for this batch

        df = spark.sql(non_negative_and_positive_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)

    # Write all numeric validation errors to the Delta error table if any
    if results:
        write_to_table(results, table_path)
        print(f"Wrote", results.count(), "DQ Errors from Format Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

### Column length validation (Min, Max and accepted range)

In [ ]:
# Function to validate column length rules (min, max, range)
def column_length_checks():
    """
    Handles rules:
    - min_column_length: LENGTH(col) < accepted_min_value
    - max_column_length: LENGTH(col) > accepted_max_value
    - accept_column_length_range: LENGTH(col) NOT BETWEEN min & max
    """

    # Select metadata for column length validation from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name",
        "source", "alternate_key_expr", "primary_key_expr", "table_display_name", "column_display_name","test", "current_value_expr",
        "created_date_field", "created_by", "error_message", "rule_type", "dq_metric", "priority","storage_type", "condition",
        "accepted_min_value", "accepted_max_value", "unaccepted_min_value", "unaccepted_max_value"
    ).filter("rule_name IN ('Accepted Min Length Check', 'Accepted Max Length Check', 'Accepted Length Range Check', 'Unaccepted Length Range Check')").collect()
    
    # Initialize empty DataFrame to store results
    results = None

    # Process rules in batches to avoid large SQL queries
    for group in batch(meta, 10):
        # Build SQL query for the batch applying column length rules
        accepted_column_length_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND LENGTH({m.column_name}) {
                        'NOT BETWEEN ' + str(m.accepted_min_value) + ' AND ' + str(m.accepted_max_value) 
                        if m.rule_name == 'Accepted Length Range Check'
                        else 'BETWEEN ' + str(m.unaccepted_min_value) + ' AND ' + str(m.unaccepted_max_value)
                        if m.rule_name == 'Unaccepted Length Range Check'
                        else '< ' + str(m.accepted_min_value)
                        if m.rule_name == 'Accepted Min Length Check'
                        else '> ' + str(m.accepted_max_value)
                    }
               """
            for m in group
        ])

        # Execute SQL for this batch
        df = spark.sql(accepted_column_length_sql)
        # Combine batch results with previous results
        results = df if results is None else results.unionByName(df)
        
    # Write all column length validation errors to the Delta error table
    write_to_table(results, table_path)
    print(f"Wrote", results.count(), "DQ Errors from Column Length Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Duration Checks Between Dates and Timestamps

In [ ]:
# Function to validate duration rules between two timestamp columns
def duration_checks():
    """
    Handles rules:
    - min_column_length: LENGTH(col) < accepted_min_value
    - max_column_length: LENGTH(col) > accepted_max_value
    - accept_column_length_range: LENGTH(col) NOT BETWEEN min & max
    """
    # Select metadata for duration validation from dq_config
    meta = dq_config.select(
        "alternate_key_cols", "storage_name", "schema_name", "table_name", "column_name", "rule_name", "source", "alternate_key_expr",
        "primary_key_expr", "table_display_name", "column_display_name","test", "current_value_expr", "created_date_field", "created_by" ,"error_message",
        "rule_type", "dq_metric", "priority","storage_type", "condition", "accepted_min_value", "accepted_max_value", "unaccepted_min_value", "unaccepted_max_value"
    ).filter("rule_name IN ('Accepted Min Duration Check', 'Accepted Max Duration Check', 'Accepted Duration Range Check', 'Unaccepted Duration Range Check')").collect()
    
    # Initialize empty DataFrame to store results
    results = None

    for group in batch(meta, 10):
        # Build SQL query for the batch applying duration rules
        durations_sql = "\nUNION ALL\n".join([
            f"""
                SELECT
                    md5(concat_ws('',
                        CAST({m.alternate_key_cols} AS STRING),
                        '{m.storage_name}',
                        '{m.schema_name}',
                        '{m.table_name}',
                        '{m.column_name}',
                        '{m.test}',
                        '{m.rule_name}',
                        '{m.source}')
                    )                                       AS surrogate_key,
                    CAST({m.alternate_key_expr} AS STRING)  AS alternate_key,
                    CAST({m.primary_key_expr}   AS STRING)  AS primary_key,
                    '{m.table_display_name}'                AS table_name,
                    '{m.column_display_name}'               AS column_name,
                    '{m.source}'                            AS source,
                    '{m.test}'                              AS test,
                    CAST({m.current_value_expr} AS STRING)  AS current_value,
                    DATE_FORMAT(
                        {m.created_date_field}, 'yyyy-MM-dd'
                    )                                       AS created_on,
                    CAST({m.created_by}          AS STRING) AS created_by,
                    '{m.error_message}'                     AS error_message,
                    '{m.rule_type}'                         AS rule_type,
                    '{m.dq_metric}'                         AS dq_metric,
                    '{m.priority}'                          AS priority,
                    CURRENT_TIMESTAMP()                     AS refreshed_at
                FROM
                    {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
                WHERE
                    {m.condition}
                    AND (unix_timestamp({m.column_name.split(',')[0].strip()}) - 
                            unix_timestamp({m.column_name.split(',')[1].strip()})
                        ) / 60 {
                        'NOT BETWEEN ' + str(m.accepted_min_value) + ' AND ' + str(m.accepted_max_value) 
                        if m.rule_name == 'Accepted Duration Range Check'
                        else 'BETWEEN ' + str(m.unaccepted_min_value) + ' AND ' + str(m.unaccepted_max_value)
                        if m.rule_name == 'Unaccepted Duration Range Check'
                        else '< ' + str(m.accepted_min_value)
                        if m.rule_name == 'Accepted Min Duration Check'
                        else '> ' + str(m.accepted_max_value)
                    }
               """
            for m in group
        ])

        # Execute SQL for this batch
        df = spark.sql(durations_sql)
        results = df if results is None else results.unionByName(df)

    # Write all column length validation errors to the Delta error table
    write_to_table(results, table_path)
    print(f"Wrote", results.count(), "DQ Errors from Duration Checks")

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Datatype check

In [ ]:
############

# Need Testing

############


# def data_type_checks():
#     """
#     Validates that column values match the expected data type.
#     Fails if TRY_CAST(column AS expected_data_type) returns NULL.
#     """

#     meta = dq_config.select(
#         "storage_name", "schema_name", "table_name", "column_name",
#         "condition", "error_message", "alternate_key_cols", "primary_key_cols",
#         "current_value_expr", "rule_type", "rule_name", "dq_metric", "expected_data_type"
#     ).filter("rule_name = 'data_type' AND expected_data_type IS NOT NULL").collect()

#     if not meta:
#         print("No data type rules found.")
#         return

#     results = None

#     for group in batch(meta, 10):
#         sql_queries = []
#         for m in group:
#             sql_queries.append(f"""
#                 SELECT
#                     md5(concat_ws('',
#                         CAST({m.alternate_key_cols} AS STRING),
#                         '{m.storage_name}',
#                         '{m.schema_name}',
#                         '{m.table_name}',
#                         '{m.column_name}',
#                         '{m.rule_name}'
#                     )) AS surrogate_key,
#                     CAST({m.alternate_key_cols} AS STRING) AS alternate_key,
#                     CAST({m.primary_key_cols} AS STRING) AS primary_key,
#                     '{m.table_name}' AS table_name,
#                     '{m.column_display_name}'                       AS column_name,
#                     '{m.storage_name}' AS source,
#                     '{m.rule_name}' AS test,
#                     CAST({m.current_value_expr} AS STRING) AS current_value,
#                     current_date() AS created_on,
#                     CAST({m.created_by}          AS STRING) AS created_by,
#                     '{m.error_message}' AS error_message,
#                     '{m.rule_type}' AS rule_type,
#                     '{m.dq_metric}' AS dq_metric,
#                     CURRENT_TIMESTAMP() AS refreshed_at
#                 FROM
#                     {f'{m.storage_name}.{m.schema_name}.{m.table_name}' if m.storage_type == "lakehouse" else f'{m.storage_name}_{m.schema_name}_{m.table_name}_view'}
#                 WHERE
#                     {m.condition}
#                     AND {m.column_name} IS NOT NULL
#                     AND TRY_CAST({m.column_name} AS {m.expected_data_type}) IS NULL
#             """)

#         batch_sql = "\nUNION ALL\n".join(sql_queries)
#         df = spark.sql(batch_sql)
#         results = df if results is None else results.unionByName(df)

#     if results:
#         write_to_table(results, table_path)
#         print(f"Wrote", results.count(), "DQ Errors from datatype Checks")
#     else:
#         print("✅ No data type mismatches found.")

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
def run_all_checks():
    """
    Executes all data quality (DQ) validation functions sequentially.
    Each function checks specific rules and writes violations to the Delta error table.
    """  
    null_and_not_checks()                    
    accepted_and_unaccepted_values_checks()  
    accepted_range_check()                   
    uniqueness_check()                       
    non_negative_and_positive_values_checks()
    column_length_checks()                   
    accepted_string_check()                  
    accepted_prefix_and_suffix_checks       
    format_checks()
    # data_type_checks()
    duration_checks()                        

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
def write_to_table(results_df, table_path):
    """
    Writes a Spark DataFrame to a Delta table.
    
    Components:
    - format("delta")       : Use Delta Lake format
    - mode("append")        : Append new records to existing table
    - option("mergeSchema") : Automatically merge schema changes
    - save(table_path)      : Save to specified Delta table path
    """
    results_df.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save(table_path)

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
run_all_checks()

StatementMeta(, , -1, Cancelled, , Cancelled)

#### Job Split Sum Accuracy Check (To 100)

In [ ]:
job_split_checks_sql = """
    /* ------------------------------------------------------------------------------------------------------------------------------------
        DQ Rule: Job Split Percentage check for Techs
            For non-canceled jobs and appointments, if the sum of active assigned technician's job splits does not equal 100%, flag the job 
    ---------------------------------------------------------------------------------------------------------------------------------------- */

    SELECT
        MD5(CONCAT(
            CAST(js.job_id AS STRING),
            'lh_core',
            'Operations',
            'Job Splits',
            'Technician Split %',
            'Technician Split Accuracy Check (0 - 100)',
            'Accepted Range Check'
            'Source 1')
        )                                           AS surrogate_key,
        CAST(js.job_id     AS STRING)               AS alternate_key,
        CAST(j.job_number  AS STRING)               AS primary_key,
        'Job-Splits'                                AS table_name,
        'Split Percentage'                          AS column_name,
        'Source 1'                                  AS source,
        'Technician Split Accuracy Check (0 - 100)' AS test,
        CAST(
            ROUND(SUM(CAST(split AS DECIMAL(8, 2))), 2
        ) AS STRING)                                AS current_value,
        DATE_FORMAT(MAX(j.created_on), 'yyyy-MM-dd')AS created_on,
        STRING(MAX(j.created_by_id))                AS created_by,            
        CASE
            WHEN ROUND(SUM(CAST(split AS DECIMAL(8, 2))), 2) < 0      THEN 'Invalid split: Negative percentage detected'
            WHEN ROUND(SUM(CAST(split AS DECIMAL(8, 2))), 2) < 99.5   THEN 'Inaccurate Split: Percentage total under 100%'
            WHEN ROUND(SUM(CAST(split AS DECIMAL(8, 2))), 2) > 100.05 THEN 'Invalid Split: Percentage total exceeds 100%'
            ELSE NULL
        END                                  AS error_message,
        'Business'                           AS rule_type,
        'Accuracy'                           AS dq_metric,
        'High'                               AS priority,
        CURRENT_TIMESTAMP()                  AS refreshed_at
    FROM
        lh_core.operations.job_splits        AS js
    LEFT JOIN
        (
            SELECT DISTINCT
                job_id,
                technician_id
            FROM
                lh_core.operations.appointment_assignments
            WHERE
                active = 'True'
                AND status IN ('Done', 'Completed')
        )                                   AS aa
            ON  js.technician_id = aa.technician_id
            AND js.job_id        = aa.job_id
    LEFT JOIN
        lh_core.jpm.jobs                    AS j 
            ON  js.job_id = j.id 
            AND js.entity_name = j.entity_name
    WHERE
        j.job_status <> 'Canceled'
    GROUP BY
        js.job_id,
        j.job_number
    HAVING
        SUM(CAST(split AS DECIMAL(8, 2))) NOT BETWEEN 0.00 AND 100.05
"""
# Write the DQ Errors to the table
results = spark.sql(job_split_checks_sql)
write_to_table(results, table_path)
print("Wrote ", results.count(), " Errors from Job Splits Accuracy Check")

StatementMeta(, , -1, Cancelled, , Cancelled)

In [ ]:
# Stop the Spark session
mssparkutils.session.stop()

StatementMeta(, , -1, Cancelled, , Cancelled)